# Блокнот 10. Нейросеть на чистом NumPy

Курс «Машинное обучение с нуля», модуль 10. Разделы: 1 — уроки 10.2–10.3, 2 — урок 10.4, 3 — урок 10.5, 4 — урок 10.6. Выполняйте ячейки сверху вниз; там, где `...  # ваш код`, допишите решение.

Сеть: вход → скрытый слой с ReLU → один выход с сигмоидой; потеря — бинарная кросс-энтропия.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(0)


def relu(z):
    return np.maximum(0, z)


def sigmoid(z):
    return 1 / (1 + np.exp(-z))

## 1. Прямой проход (уроки 10.2–10.3)

**Задание 1.** Допишите прямой проход: `Z1 = X @ W1 + b1`, `H = relu(Z1)`, `p = sigmoid(H @ W2 + b2)`. Верните `p` и словарь промежуточных значений — они понадобятся обратному проходу.

In [ ]:
def forward(X, params):
    W1, b1, W2, b2 = params["W1"], params["b1"], params["W2"], params["b2"]
    Z1, H, p = ..., ..., ...  # ваш код
    return p, {"X": X, "Z1": Z1, "H": H}

In [ ]:
xor = {"W1": np.array([[1.0, 1.0], [1.0, 1.0]]), "b1": np.array([0.0, -1.0]),
       "W2": np.array([4.0, -8.0]), "b2": -2.0}
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float)
p, _ = forward(X_xor, xor)
print("XOR:", p.round(3))
assert np.allclose((p > 0.5).astype(int), [0, 1, 1, 0])

## 2. Обратный проход и проверка градиента (урок 10.4)

Потеря — средняя бинарная кросс-энтропия. Её градиент по логиту выхода — `(p - y) / n` (как у логистической регрессии).

**Задание 2.** Допишите обратный проход: градиенты `W2`, `b2`, затем через ReLU — `W1`, `b1`.

In [ ]:
def loss_fn(p, y):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))


def backward(p, y, cache, params):
    n = len(y)
    dout = (p - y) / n                       # dL/d(логит выхода), форма (n,)
    dW2, db2, dW1, db1 = ..., ..., ..., ...  # ваш код
    return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}

In [ ]:
def init(d, h, scale):
    return {"W1": rng.normal(0, scale, (d, h)), "b1": np.zeros(h), "W2": rng.normal(0, scale, h), "b2": 0.0}


X_small = rng.normal(size=(5, 2))
y_small = np.array([0, 1, 1, 0, 1.0])
params = init(2, 4, 1.0)
p, cache = forward(X_small, params)
grads = backward(p, y_small, cache, params)
eps = 1e-6
for name in ("W1", "b1", "W2"):
    num = np.zeros_like(params[name])
    for idx in np.ndindex(params[name].shape):
        old = params[name][idx]
        params[name][idx] = old + eps
        lp = loss_fn(forward(X_small, params)[0], y_small)
        params[name][idx] = old - eps
        lm = loss_fn(forward(X_small, params)[0], y_small)
        params[name][idx] = old
        num[idx] = (lp - lm) / (2 * eps)
    err = np.abs(num - grads[name]).max()
    print(f"{name}: максимальное расхождение с численным градиентом {err:.2e}")
    assert err < 1e-6
print("Обратный проход верен!")

## 3. Обучение на полумесяцах (урок 10.5)

Инициализация He: стандартное отклонение $\sqrt{2 / n_{\text{вход}}}$. Сравним с нулями.

In [ ]:
X, y = make_moons(600, noise=0.25, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.5, random_state=0)


def train(params, X, y, lr=0.5, epochs=2000, weight_decay=0.0):
    history = []
    for _ in range(epochs):
        p, cache = forward(X, params)
        grads = backward(p, y, cache, params)
        for k in params:
            params[k] = params[k] - lr * (grads[k] + weight_decay * params[k])
        history.append(loss_fn(p, y))
    return params, history


def accuracy(params, X, y):
    return np.mean((forward(X, params)[0] > 0.5) == y)


he = {"W1": rng.normal(0, np.sqrt(2 / 2), (2, 32)), "b1": np.zeros(32), "W2": rng.normal(0, np.sqrt(2 / 32), 32), "b2": 0.0}
zeros = {"W1": np.zeros((2, 32)), "b1": np.zeros(32), "W2": np.zeros(32), "b2": 0.0}
he, hist_he = train(he, X_tr, y_tr)
zeros, hist_zero = train(zeros, X_tr, y_tr)
print(f"He: тест {accuracy(he, X_te, y_te):.3f}; нули: тест {accuracy(zeros, X_te, y_te):.3f}")
plt.plot(hist_he, label="He")
plt.plot(hist_zero, label="нули")
plt.xlabel("эпоха")
plt.ylabel("потеря")
plt.legend()
plt.show()
assert accuracy(he, X_te, y_te) > 0.9

С нулями все скрытые нейроны одинаковы, а градиенты `W1` и вовсе нулевые: сеть остаётся линейной моделью на одном выходе. Граница обученной сети:

In [ ]:
xx, yy = np.meshgrid(np.linspace(-1.6, 2.6, 200), np.linspace(-1.2, 1.7, 200))
zz = forward(np.c_[xx.ravel(), yy.ravel()], he)[0].reshape(xx.shape)
plt.contourf(xx, yy, zz, levels=[0, 0.5, 1], alpha=0.3)
plt.scatter(*X_tr.T, c=y_tr, s=8, cmap="coolwarm")
plt.title("Граница сети 2 → 32 → 1 на NumPy")
plt.show()

## 4. Weight decay на маленькой выборке (урок 10.6)

**Задание 3.** Обучите сеть 2 → 256 → 1 на 100 зашумлённых точках без штрафа и с `weight_decay=0.01`. Сравните точность на тесте и сумму квадратов весов.

In [ ]:
Xs, ys = make_moons(1100, noise=0.35, random_state=3)
Xs_tr, Xs_te, ys_tr, ys_te = train_test_split(Xs, ys, train_size=100, random_state=0)
for wd in (0.0, 0.01):
    net = {"W1": rng.normal(0, 1.0, (2, 256)), "b1": np.zeros(256), "W2": rng.normal(0, np.sqrt(2 / 256), 256), "b2": 0.0}
    net = ...  # ваш код: train с нужным weight_decay
    w2 = (net["W1"] ** 2).sum() + (net["W2"] ** 2).sum()
    print(f"weight_decay {wd}: обучение {accuracy(net, Xs_tr, ys_tr):.2f}, тест {accuracy(net, Xs_te, ys_te):.3f}, сумма w² {w2:.0f}")